- https://amazon-reviews-2023.github.io/

In [1]:
import os
import pandas as pd

In [2]:
PATH = "./data"

In [3]:
os.makedirs(os.path.join(PATH, "Baby_Products_meta_images"), exist_ok=True)

In [23]:
import pandas as pd
import os

def get_chunk_size(filename, chunksize=200_000):
  # Ensure PATH is defined (it's already in the notebook state)
  filepath = os.path.join(PATH, filename)

  chunk_count = 0
  try:
      for chunk in pd.read_json(filepath, lines=True, chunksize=chunksize):
          chunk_count += 1
      print(f"Total number of chunks: {chunk_count}")
  except FileNotFoundError:
      print(f"Error: File not found at {filepath}")
  except Exception as e:
      print(f"An error occurred: {e}")

get_chunk_size("meta_Baby_Products.jsonl", chunksize=50_000)

Total number of chunks: 5


In [34]:

# Đảm bảo sử dụng index toàn cục bằng cách không thiết lập index=False
reader = pd.read_json(os.path.join(PATH, "meta_Baby_Products.jsonl"), lines=True, chunksize=50_000)

cnt = 0
for chunk_idx, df in enumerate(reader):
    cnt += 1
    print(chunk_idx, cnt)
    # 1. Bỏ qua các chunk trước start_chunk
    if cnt == 5:
        print("last")
        break


    # print(f"Processing chunk {chunk_idx} with {len(df)} records")

0 1
1 2
2 3
3 4
4 5
last


In [28]:
df.iloc[0]['images']

[{'thumb': 'https://m.media-amazon.com/images/I/41+SfWrfPkL._SS40_.jpg',
  'large': 'https://m.media-amazon.com/images/I/41+SfWrfPkL.jpg',
  'variant': 'MAIN',
  'hi_res': 'https://m.media-amazon.com/images/I/6173uqKVzfL._SL1300_.jpg'},
 {'thumb': 'https://m.media-amazon.com/images/I/41UaLOOlyFL._SS40_.jpg',
  'large': 'https://m.media-amazon.com/images/I/41UaLOOlyFL.jpg',
  'variant': 'PT01',
  'hi_res': 'https://m.media-amazon.com/images/I/61yMQ5y6OAL._SL1300_.jpg'},
 {'thumb': 'https://m.media-amazon.com/images/I/41cldZMHt7L._SS40_.jpg',
  'large': 'https://m.media-amazon.com/images/I/41cldZMHt7L.jpg',
  'variant': 'PT02',
  'hi_res': 'https://m.media-amazon.com/images/I/61oyGcdrfqL._SL1300_.jpg'},
 {'thumb': 'https://m.media-amazon.com/images/I/4178ufyvQ1L._SS40_.jpg',
  'large': 'https://m.media-amazon.com/images/I/4178ufyvQ1L.jpg',
  'variant': 'PT03',
  'hi_res': 'https://m.media-amazon.com/images/I/614R+LDIVrL._SL1300_.jpg'},
 {'thumb': 'https://m.media-amazon.com/images/I/41S9

### Song song

In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os, json, ast, time, shutil, random, logging, requests
import numpy as np
import pandas as pd
from urllib.parse import urlparse

# =========================
# CONFIG
# =========================
MAX_THREADS = 30        # Số luồng SONG SONG cho CÁC SẢN PHẨM (product)

URL_PRIORITIES = ['hi_res', 'large', 'thumb']
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Accept': 'image/webp,image/*,*/*;q=0.8'
}

# 🆕 CẤU HÌNH RATE LIMITING
MIN_DOWNLOAD_DELAY = 1.0 
MAX_DOWNLOAD_DELAY = 2.0 


# =========================
# Logging – DEBUG vào File, ERROR ra Console
# =========================
import logging # Đảm bảo import logging

def setup_logging(log_file):
    # 1. Khởi tạo Logger chính
    logger = logging.getLogger("ImageDownloader")
    
    if not logger.handlers: 
        # Đặt mức độ logger chính là DEBUG (thấp nhất) để bắt được tất cả các thông báo
        logger.setLevel(logging.DEBUG)
        
        formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

        # --- 2. File Handler (Ghi vào File) ---
        file_handler = logging.FileHandler(log_file, 'a', 'utf-8')
        # Đặt mức độ log cho File Handler là DEBUG
        file_handler.setLevel(logging.DEBUG) 
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)
        
        # --- 3. Console Handler (Ghi ra Console/Màn hình) ---
        # Sử dụng StreamHandler để ghi ra stdout (console)
        console_handler = logging.StreamHandler()
        # Đặt mức độ log cho Console Handler là ERROR
        console_handler.setLevel(logging.ERROR) 
        console_handler.setFormatter(formatter)
        logger.addHandler(console_handler)
    
    return logger

# =========================
# Parse image list (Giữ nguyên)
# =========================
def parse_image_list(images_data):
    if isinstance(images_data, list):
        return images_data
    if isinstance(images_data, str):
        try:
            return ast.literal_eval(images_data)
        except (ValueError, SyntaxError):
            try:
                clean = images_data.replace("'", '"').replace("None", "null").replace("False", "false").replace("True", "true")
                return json.loads(clean)
            except:
                pass
    return None


# # =========================
# Download 1 image với retry (Xử lý 404 và Rate Limiting)
# =========================
def download_one_image(url, save_path, idx, logger, max_retry=3):
    logger.debug(f"[{idx}] Start download: {url}")
    
    for attempt in range(1, max_retry + 1):
        
        # RATE LIMITING: Độ trễ ngẫu nhiên trước lần thử đầu tiên
        if attempt == 1:
            time.sleep(random.uniform(MIN_DOWNLOAD_DELAY, MAX_DOWNLOAD_DELAY))
        
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            
            # XỬ LÝ LỖI VĨNH VIỄN 404
            if resp.status_code == 404:
                logger.error(f"[{idx}] FAILED FINAL (404 Not Found): {url}")
                return False # Bỏ qua retry nếu gặp 404
            
            # Kiểm tra các lỗi khác (4xx, 5xx)
            resp.raise_for_status()
            
            # Tải thành công
            with open(save_path, "wb") as f:
                f.write(resp.content)
            
            # 🆕 LOG THÔNG BÁO TẢI THÀNH CÔNG
            logger.info(f"[{idx}] SUCCESS: {url}")
            return True
            
        except Exception as e:
            logger.error(f"[{idx}] attempt {attempt}/{max_retry} → {url} : {e}")
            if attempt == max_retry:
                logger.error(f"[{idx}] FAILED FINAL: {url}")
                return False
            
            # Độ trễ tăng dần cho retry (Exponential Backoff)
            retry_delay = (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            time.sleep(retry_delay)

# =========================
# Submit download tasks cho 1 product (row) – CHẠY TUẦN TỰ CHO CÁC ẢNH
# =========================
# Đã loại bỏ tham số 'executor'
def download_row_images(idx, images_data, output_dir, logger):
    if images_data is None or images_data is np.nan:
        return True # Hoàn thành xử lý nếu không có dữ liệu

    if not isinstance(images_data, list):
        images_data = parse_image_list(images_data)

    if not images_data:
        return True # Hoàn thành xử lý nếu danh sách ảnh rỗng

    # Đảm bảo thư mục sản phẩm tồn tại
    product_dir = os.path.join(output_dir, str(idx))
    os.makedirs(product_dir, exist_ok=True) 

    success_count = 0
    
    # Vòng lặp tuần tự cho từng ảnh trong sản phẩm
    for i, img in enumerate(images_data):
        url = next((img.get(p) for p in URL_PRIORITIES if img.get(p)), None)
        
        if not url:
            continue
            
        name = img.get("variant", f"img_{i:02d}")
        name = name.replace("/", "_").replace("\\", "_").replace(":", "_").replace("*", "_").replace("?", "_")
        
        ext = os.path.splitext(urlparse(url).path)[1] or ".jpg"
        if not ext.startswith("."):
             ext = "." + ext

        save_path = os.path.join(product_dir, f"{name}{ext}")
        
        # CHỈ TẢI NẾU TỆP ĐÍCH CHƯA TỒN TẠI (Resume)
        if os.path.exists(save_path):
            success_count += 1
            continue

        # Gọi download_one_image trực tiếp (chạy tuần tự)
        if download_one_image(url, save_path, idx, logger):
            success_count += 1
    
    # Trả về True/False để cho biết sản phẩm này đã được xử lý xong
    return success_count == len(images_data)


# =========================
# Kiểm tra sự tồn tại và số lượng ảnh đã hoàn thành
# =========================
def is_product_complete(idx, images_data, chunk_dir):
    """Kiểm tra nếu thư mục sản phẩm tồn tại và có đủ số lượng ảnh."""
    product_dir = os.path.join(chunk_dir, str(idx))

    # 1. Kiểm tra nếu không có dữ liệu ảnh (được coi là hoàn thành)
    if images_data is None or images_data is np.nan:
        return True
    
    if not isinstance(images_data, list):
        images_data = parse_image_list(images_data)
    
    if not images_data:
        return True # Không có ảnh để tải

    expected_count = len(images_data)
    
    # 2. Kiểm tra nếu folder không tồn tại, chắc chắn chưa hoàn thành
    if not os.path.exists(product_dir):
        return False
    
    # 3. Kiểm tra số lượng file trong folder
    actual_count = len(os.listdir(product_dir))
    
    # Lưu ý: Việc kiểm tra số lượng file không đảm bảo tên file khớp, 
    # nhưng đây là cách kiểm tra nhanh (quick check)
    return actual_count >= expected_count


# =========================
# Process 1 chunk (ĐÃ SỬA ĐỔI ĐỂ KIỂM TRA ĐỦ HÌNH TRƯỚC VÀ DÙNG LOG)
# =========================
def process_one_chunk(chunk_idx, df, output_root, log_file):
    # Khởi tạo logger cho chunk này
    logger = setup_logging(log_file) 
    
    chunk_dir = os.path.join(output_root, f"chunk_{chunk_idx}")
    os.makedirs(chunk_dir, exist_ok=True)
    
    all_tasks = []
    
    # Khởi tạo ThreadPoolExecutor: MAX_THREADS là số lượng SẢN PHẨM được xử lý song song
    with ThreadPoolExecutor(max_workers=MAX_THREADS) as pool:
        
        # 1. Lặp qua TỪNG SẢN PHẨM (tuần tự) và submit tác vụ xử lý sản phẩm
        for idx, row in df.iterrows():
            
            # KIỂM TRA NẾU SẢN PHẨM ĐÃ HOÀN THÀNH TẢI XUỐNG
            if is_product_complete(idx, row["images"], chunk_dir):
                logger.debug(f"[{idx}] Folder tồn tại và đủ hình, bỏ qua tạo task.")
                continue # BỎ QUA tạo task nếu đã hoàn thành
            
            # Gửi toàn bộ việc xử lý một sản phẩm (tải ảnh tuần tự) vào một luồng
            future = pool.submit(download_row_images, idx, row["images"], chunk_dir, logger)
            all_tasks.append(future)
            
        # 2. Đợi TẤT CẢ sản phẩm hoàn thành
        # SỬ DỤNG logger.info THAY CHO print
        logger.info(f"Chunk {chunk_idx}: Total {len(all_tasks)} product tasks submitted. Waiting...")
        
        completed_products = 0
        for future in as_completed(all_tasks):
            completed_products += 1
            try:
                future.result() 
            except Exception as e:
                logger.error(f"A product processing task failed unexpectedly: {e}")

    return chunk_idx


# =========================
# Main – chunk tuần tự, sản phẩm song song (ĐÃ THÊM end_chunk)
# =========================
def process_chunks(json_file, chunk_size=10, output_root="download_chunks",
                   log_file="error.log", start_chunk=0, end_chunk=None): # <--- THÊM end_chunk
    
    # Setup logger ở đây để ghi các thông báo tổng quan của hàm main
    main_logger = setup_logging(log_file)

    # Đảm bảo sử dụng index toàn cục bằng cách không thiết lập index=False
    reader = pd.read_json(json_file, lines=True, chunksize=chunk_size)

    for chunk_idx, df in enumerate(reader):
        
        # 1. Bỏ qua các chunk trước start_chunk
        if chunk_idx < start_chunk:
            main_logger.info(f"⏭ Skip chunk {chunk_idx}")
            continue

        # 2. DỪNG LẠI nếu đạt đến end_chunk (nếu được chỉ định)
        # Sử dụng > end_chunk vì chúng ta muốn xử lý chunk_idx == end_chunk
        if end_chunk is not None and chunk_idx > end_chunk:
            main_logger.info(f"🛑 Reached end_chunk {end_chunk}. Stopping.")
            break 

        main_logger.info(f"▶ Processing chunk {chunk_idx} (rows {df.index.min()} to {df.index.max()}) ...")
        
        start_time = time.time()
        # Trong process_chunks, chúng ta không cần truyền logger vì process_one_chunk tự setup
        idx = process_one_chunk(chunk_idx, df, output_root, log_file) 
        end_time = time.time()
        
        main_logger.info(f"✔ Chunk {idx} done in {end_time - start_time:.2f}s")

    main_logger.info("🎉 ALL DONE")


In [22]:
filepath = os.path.join(PATH, "meta_Baby_Products.jsonl")

process_chunks(filepath, chunk_size=50_000, output_root=os.path.join(PATH, "Baby_Products_meta_images"),
               log_file=os.path.join(PATH, "download_errors.log"),
               start_chunk=5, end_chunk=5)

In [19]:
import os
import tarfile

images_root = os.path.join(PATH, "Baby_Products_meta_images")
output_dir = os.path.join(PATH, "compressed_chunks")

os.makedirs(output_dir, exist_ok=True)

chunk_folders = sorted(os.listdir(images_root), key=lambda x: int(x.split('_')[1]))
chunk_folders

['chunk_2', 'chunk_4']

In [20]:
# Compress only the first 10 chunk directories to .tar.gz
for chunk_folder in chunk_folders[1:]:
    chunk_path = os.path.join(images_root, chunk_folder)
    
    if os.path.isdir(chunk_path):
        output_file = os.path.join(output_dir, f"{chunk_folder}.tar.gz")
        
        # Create tar.gz archive
        with tarfile.open(output_file, "w:gz") as tar:
            tar.add(chunk_path, arcname=chunk_folder)
        
        print(f"Compressed: {output_file}")

print("Chunks compressed successfully!")

Compressed: ./data\compressed_chunks\chunk_4.tar.gz
Chunks compressed successfully!
